# Loss comparison — figures

Reads `loss_study/results/summary.csv` (produced by `python loss_study/aggregate.py`) and draws model × loss heat-maps per metric and domain, plus bar charts of the mean rank of every loss.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")
RESULTS = Path("../results/summary.csv")
df = pd.read_csv(RESULTS)
df = df[df["loss_task"] == "regression"]
print(f"{df['loss'].nunique()} losses x {df['model_dir'].nunique()} models")
df.head()

In [ ]:
# Metric -> direction (-1: lower is better, +1: higher is better)
METRICS = {"mse": -1, "mae": -1, "bias": -1, "smape": -1, "correlation_coef": +1, "hss": +1, "hss_heavy": +1}
DOMAINS = ["austria", "conus", "korea"]

def pivot(metric, domain=None):
    sub = df[df["metric"] == metric]
    if domain is not None:
        sub = sub[sub["domain"] == domain]
    return sub.pivot_table(index="model_dir", columns="loss", values="value", aggfunc="mean")

def heatmaps(metric):
    fig, axes = plt.subplots(1, len(DOMAINS) + 1, figsize=(5 * (len(DOMAINS) + 1), 4), sharey=True)
    for ax, domain in zip(axes, DOMAINS + [None]):
        table = pivot(metric, domain)
        if metric == "bias":
            table = table.abs()
        cmap = "viridis_r" if METRICS[metric] < 0 else "viridis"
        sns.heatmap(table, annot=True, fmt=".3g", cmap=cmap, ax=ax, cbar=False)
        ax.set_title(f"{metric} — {domain or 'mean over domains'}")
        ax.set_xlabel("loss"); ax.set_ylabel("model")
    plt.tight_layout()
    return fig

for metric in METRICS:
    if metric in set(df["metric"]):
        heatmaps(metric)

In [ ]:
# Mean rank of each loss across models (1 = best), per metric
ranks = {}
for metric, direction in METRICS.items():
    if metric not in set(df["metric"]):
        continue
    table = pivot(metric)
    if metric == "bias":
        table = table.abs()
    ranks[metric] = table.rank(axis=1, ascending=(direction < 0)).mean(axis=0)
ranks = pd.DataFrame(ranks)
ranks["mean_rank"] = ranks.mean(axis=1)
ranks = ranks.sort_values("mean_rank")

fig, ax = plt.subplots(figsize=(8, 4))
ranks.drop(columns="mean_rank").plot.bar(ax=ax)
ax.set_ylabel("mean rank across models (1 = best)")
ax.set_xlabel("loss")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
ranks

In [ ]:
# Per-model view: how much does each loss change a metric relative to the MSE baseline?
metric = "mae"
table = pivot(metric)
if "mse" in table.columns:
    rel = 100 * (table.sub(table["mse"], axis=0)).div(table["mse"], axis=0)
    fig, ax = plt.subplots(figsize=(8, 4))
    sns.heatmap(rel.drop(columns="mse"), annot=True, fmt=".1f", cmap="RdBu_r", center=0, ax=ax)
    ax.set_title(f"{metric}: % change vs. MSE baseline (negative = better)")
    plt.tight_layout()